# NB18: Manuscript Figures and Tables

All publication-quality figures and tables for Paper 3.
No new computation — reads from upstream CSVs in `results/`.

Figures are saved to `manuscript/figures/` as PDF (vector) and PNG.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path

from src.database import get_session, query_profiles_as_dataframe, query_fits_as_dataframe
from src.physics import rt_model_velocity
from src.btfr import compute_mbar_for_sample, BTFR_ALPHA, BTFR_BETA
from src.utils import get_project_root

root = get_project_root()
results_dir = root / "results"
fig_dir = root / "manuscript" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

# Constants
A0 = 1.2e-10
A0_HALF = A0 / 2

# Publication style
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

# Colorblind-safe palette
C_SPARC = '#999999'
C_THINGS = '#0072B2'
C_LT = '#D55E00'
C_A0HALF = '#CC79A7'

def save_fig(fig, name):
    """Save figure as PDF and PNG."""
    fig.savefig(fig_dir / f"{name}.pdf")
    fig.savefig(fig_dir / f"{name}.png")
    print(f"Saved: {name}.pdf, {name}.png")

In [2]:
# Load all upstream data
sparc_df = pd.read_csv(results_dir / "NB01_baseline_results.csv")
things_df = pd.read_csv(results_dir / "NB09_things_per_galaxy.csv")
lt_df = pd.read_csv(results_dir / "NB13_lt_per_galaxy.csv")

# Summary tables from NB17
cross_summary = pd.read_csv(results_dir / "NB17_cross_dataset_summary.csv")
pooled_df = pd.read_csv(results_dir / "NB17_pooled_test.csv")
demographics = pd.read_csv(results_dir / "NB17_mass_demographics.csv")
constrained_summary = pd.read_csv(results_dir / "NB17_constrained_summary.csv")

# BIC data
things_bic = pd.read_csv(results_dir / "NB10_things_bic_comparison.csv")
lt_bic = pd.read_csv(results_dir / "NB12_lt_bic_comparison.csv")

# Cross-pipeline
both_resolved = pd.read_csv(results_dir / "NB08_both_resolved.csv")

# SPARC M_bar
sparc_mbar = compute_mbar_for_sample(sparc_df["galaxy_id"].tolist())
sparc_df["M_bar"] = sparc_df["galaxy_id"].map(sparc_mbar)

print("All data loaded.")

2026-04-09 23:59:21 | INFO     | src.ingest | Loaded 175 galaxies from SPARC metadata


2026-04-09 23:59:21 | INFO     | src.ingest | Loaded 175 bulge luminosities


All data loaded.


## Figure 1: RT Model Example Fit

Exemplary rotation curve fit for NGC 2403 (SPARC) showing $V_{\rm obs}$,
$V_{\rm bary}$, $V_{\rm model}$, with $R_t$ marked.

In [3]:
# Load NGC 2403 profile and fit parameters
session = get_session()
ngc2403_prof = query_profiles_as_dataframe(session, galaxy_id="NGC2403")
ngc2403_fit = query_fits_as_dataframe(session, galaxy_id="NGC2403", model_name="rational_taper")
session.close()

r = ngc2403_prof["radius_kpc"].values
v_obs = ngc2403_prof["v_obs"].values
v_err = ngc2403_prof["v_err"].values
v_bary = ngc2403_prof["v_baryon_total"].values

omega = ngc2403_fit["param1"].values[0]
Rt = ngc2403_fit["param2"].values[0]

# Compute model velocity: rt_model_velocity(radius, v_bary, omega, r_t)
r_fine = np.linspace(r.min(), r.max(), 300)
v_bary_interp = np.interp(r_fine, r, v_bary)
v_model = rt_model_velocity(r_fine, v_bary_interp, omega, Rt)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(r, v_obs, yerr=v_err, fmt='o', ms=4, color='k', alpha=0.6,
            label=r'$V_{\rm obs}$', zorder=3)
ax.plot(r, np.abs(v_bary), '--', color='#E69F00', lw=1.5,
        label=r'$|V_{\rm bary}|$')
ax.plot(r_fine, v_model, '-', color=C_THINGS, lw=2.5,
        label=r'$V_{\rm RT}$')
ax.axvline(Rt, color=C_A0HALF, ls=':', lw=1.5,
           label=rf'$R_t = {Rt:.1f}$ kpc')

ax.set_xlabel('Radius (kpc)')
ax.set_ylabel(r'$V$ (km s$^{-1}$)')
ax.set_title('NGC 2403 (SPARC)')
ax.legend(loc='lower right', framealpha=0.9)
ax.set_xlim(0, None)
ax.set_ylim(0, None)

save_fig(fig, "fig01_rt_model_example")
plt.show()

Saved: fig01_rt_model_example.pdf, fig01_rt_model_example.png


C:\Users\schneider\AppData\Local\Temp\ipykernel_122408\824433595.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 2: SPARC g(Rt) Distribution

In [4]:
fig, ax = plt.subplots(figsize=(5, 4))

log_g_sparc = np.log10(sparc_df["g_Rt_mks"].values)
ax.hist(log_g_sparc, bins=20, color=C_SPARC, edgecolor='white', alpha=0.85)
ax.axvline(np.log10(A0_HALF), color=C_A0HALF, ls='--', lw=2,
           label=r'$a_0/2$')
ax.axvline(np.median(log_g_sparc), color='k', ls='-', lw=1.5,
           label=f'Median ({np.median(log_g_sparc):.2f})')

ax.set_xlabel(r'$\log_{10}\, g(R_t)$ [m s$^{-2}$]')
ax.set_ylabel('Count')
ax.set_title(f'SPARC Resolved Sample (N={len(sparc_df)})')
ax.legend()

save_fig(fig, "fig02_sparc_grt_distribution")
plt.show()

Saved: fig02_sparc_grt_distribution.pdf, fig02_sparc_grt_distribution.png

C:\Users\schneider\AppData\Local\Temp\ipykernel_122408\2719427727.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 3: g(Rt) vs M_bar — Key Figure

The central figure of the paper. Shows all three datasets on the same
$g(R_t)$ vs $M_{\rm bar}$ plane with the fixed BTFR trend line.

In [5]:
fig, ax = plt.subplots(figsize=(8, 5.5))

# SPARC
sparc_valid = sparc_df.dropna(subset=["M_bar", "g_Rt_mks"])
ax.scatter(np.log10(sparc_valid["M_bar"]), np.log10(sparc_valid["g_Rt_mks"]),
           c=C_SPARC, s=20, alpha=0.5, marker='o', label=f'SPARC (N={len(sparc_valid)})',
           zorder=2)

# THINGS
ax.scatter(np.log10(things_df["M_bar"]), np.log10(things_df["g_Rt_mks"]),
           c=C_THINGS, s=60, marker='s', edgecolors='k', linewidths=0.5,
           label=f'THINGS (N={len(things_df)})', zorder=4)

# LITTLE THINGS
ax.scatter(np.log10(lt_df["M_bar"]), np.log10(lt_df["g_Rt_mks"]),
           c=C_LT, s=60, marker='^', edgecolors='k', linewidths=0.5,
           label=f'LITTLE THINGS (N={len(lt_df)})', zorder=4)

# BTFR trend line
m_range = np.linspace(6, 12, 100)
g_trend = BTFR_ALPHA * m_range + BTFR_BETA
ax.plot(m_range, g_trend, 'k-', lw=1.5, alpha=0.7,
        label=rf'BTFR trend ($\alpha={BTFR_ALPHA}$)')

# a0/2 horizontal line
ax.axhline(np.log10(A0_HALF), color=C_A0HALF, ls='--', lw=2,
           label=r'$a_0/2$', zorder=1)

ax.set_xlabel(r'$\log_{10}\, M_{\rm bar}$ [$M_\odot$]')
ax.set_ylabel(r'$\log_{10}\, g(R_t)$ [m s$^{-2}$]')
ax.legend(loc='upper left', framealpha=0.9, fontsize=9)
ax.set_xlim(6, 12)
ax.set_ylim(-12, -8.5)

save_fig(fig, "fig03_grt_vs_mbar_all")
plt.show()

Saved: fig03_grt_vs_mbar_all.pdf, fig03_grt_vs_mbar_all.png


C:\Users\schneider\AppData\Local\Temp\ipykernel_122408\542292607.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 4: BTFR Residual Distributions (3-panel)

In [6]:
from src.btfr import compute_btfr_residuals

# Compute SPARC residuals
sparc_valid = sparc_df.dropna(subset=["M_bar", "g_Rt_mks"])
sparc_resid = compute_btfr_residuals(sparc_valid["g_Rt_mks"].values,
                                      sparc_valid["M_bar"].values)

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

xlim = (-1.2, 1.2)
bins = np.linspace(xlim[0], xlim[1], 25)

datasets = [
    (sparc_resid, 'SPARC', C_SPARC, 98, 0.654),
    (things_df["residual"].values, 'THINGS', C_THINGS, 8, 0.547),
    (lt_df["residual"].values, 'LITTLE THINGS', C_LT, 14, 0.049),
]

for ax, (resid, name, color, n, p) in zip(axes, datasets):
    ax.hist(resid, bins=bins, color=color, edgecolor='white', alpha=0.85)
    med = np.median(resid)
    ax.axvline(med, color='k', ls='-', lw=1.5)
    ax.axvline(0, color=C_A0HALF, ls='--', lw=1.5)
    # +/-10% corridor
    ax.axvspan(-0.041, 0.041, alpha=0.15, color='green')
    ax.set_title(f'{name} (N={n})')
    ax.set_xlim(xlim)
    ax.text(0.95, 0.92, f'med={med:+.3f}\np={p:.3f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

axes[0].set_ylabel('Count')
axes[1].set_xlabel(r'$\Delta \log_{10}\, g(R_t)$ [dex]')

fig.tight_layout()
save_fig(fig, "fig04_btfr_residuals")
plt.show()

Saved: fig04_btfr_residuals.pdf, fig04_btfr_residuals.png


C:\Users\schneider\AppData\Local\Temp\ipykernel_122408\4204920307.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 5: Constrained Model BIC Comparison

In [7]:
# Get SPARC delta_BIC from database
session = get_session()
sparc_con = query_fits_as_dataframe(session, model_name="constrained_rt")
sparc_con = sparc_con[~sparc_con["galaxy_id"].str.contains("THINGS", na=False)]
sparc_free = query_fits_as_dataframe(session, model_name="rational_taper")
sparc_free = sparc_free[~sparc_free["galaxy_id"].str.contains("THINGS", na=False)]
session.close()

sparc_con_ok = sparc_con[sparc_con["converged"]][["galaxy_id", "bic"]]
sparc_bic_m = sparc_free[["galaxy_id", "bic"]].merge(
    sparc_con_ok, on="galaxy_id", suffixes=("_free", "_con")
)
sparc_bic_m["delta_bic"] = sparc_bic_m["bic_con"] - sparc_bic_m["bic_free"]

fig, ax = plt.subplots(figsize=(6, 4))

positions = [0, 1, 2]
labels = ['SPARC', 'THINGS', 'LITTLE THINGS']
colors = [C_SPARC, C_THINGS, C_LT]
bic_data = [
    sparc_bic_m["delta_bic"].values,
    things_bic["delta_bic"].values,
    lt_bic["delta_bic"].values,
]

for pos, data, color, label in zip(positions, bic_data, colors, labels):
    parts = ax.violinplot([data], positions=[pos], showmedians=True, showextrema=False)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    parts['cmedians'].set_color('k')
    # Add jittered points
    jitter = np.random.default_rng(42).uniform(-0.15, 0.15, len(data))
    ax.scatter(pos + jitter, data, c=color, s=15, alpha=0.6, zorder=3)

ax.axhline(0, color='k', ls='--', lw=0.8, alpha=0.5)
ax.axhline(2, color='gray', ls=':', lw=0.8, alpha=0.5, label=r'$|\Delta$BIC$| = 2$ (threshold)')
ax.axhline(-2, color='gray', ls=':', lw=0.8, alpha=0.5)

ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.set_ylabel(r'$\Delta$BIC (constrained $-$ free)')
ax.set_title('Constrained Model: Decisively Rejected Everywhere')
ax.legend(loc='upper left', fontsize=9)

# Add median annotations
for pos, data, label in zip(positions, bic_data, labels):
    med = np.median(data)
    ax.text(pos, med + max(data)*0.05, f'{med:+.0f}', ha='center', fontsize=9, fontweight='bold')

save_fig(fig, "fig05_constrained_bic")
plt.show()

Saved: fig05_constrained_bic.pdf, fig05_constrained_bic.png


C:\Users\schneider\AppData\Local\Temp\ipykernel_122408\3552048728.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 6: Cross-Pipeline Validation

In [8]:
fig, ax = plt.subplots(figsize=(5, 5))

sparc_g_vals = np.log10(both_resolved["sparc_g_mks"].values)
things_g_vals = np.log10(both_resolved["things_g_mks"].values)

ax.scatter(sparc_g_vals, things_g_vals, c=C_THINGS, s=80, marker='s',
           edgecolors='k', linewidths=0.5, zorder=3)

# Label each galaxy
for _, row in both_resolved.iterrows():
    ax.annotate(row["galaxy_id"], (np.log10(row["sparc_g_mks"]),
                np.log10(row["things_g_mks"])),
               textcoords='offset points', xytext=(5, 5), fontsize=8)

# 1:1 line
lim = [min(sparc_g_vals.min(), things_g_vals.min()) - 0.1,
       max(sparc_g_vals.max(), things_g_vals.max()) + 0.1]
ax.plot(lim, lim, 'k--', lw=1, alpha=0.5, label='1:1')

ax.set_xlabel(r'SPARC $\log_{10}\, g(R_t)$ [m s$^{-2}$]')
ax.set_ylabel(r'THINGS $\log_{10}\, g(R_t)$ [m s$^{-2}$]')
ax.set_title(f'Cross-Pipeline Validation (N={len(both_resolved)})')
ax.legend()
ax.set_aspect('equal')
ax.set_xlim(lim)
ax.set_ylim(lim)

# Annotate median offset
med_delta = np.median(both_resolved["delta_log_g"].values)
ax.text(0.05, 0.92, f'Median $\\Delta$ = {med_delta:+.3f} dex\np = 0.812',
        transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

save_fig(fig, "fig06_cross_pipeline")
plt.show()

Saved: fig06_cross_pipeline.pdf, fig06_cross_pipeline.png


C:\Users\schneider\AppData\Local\Temp\ipykernel_122408\3428632670.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 7: Pooled Test Visualization

In [9]:
fig, ax = plt.subplots(figsize=(7, 4))

things_pool = pooled_df[pooled_df["dataset"] == "THINGS"]
lt_pool = pooled_df[pooled_df["dataset"] == "LITTLE_THINGS"]

# Strip plot
y_things = np.random.default_rng(42).uniform(0.8, 1.2, len(things_pool))
y_lt = np.random.default_rng(43).uniform(1.8, 2.2, len(lt_pool))

ax.scatter(things_pool["residual"], y_things, c=C_THINGS, s=60, marker='s',
           edgecolors='k', linewidths=0.5, label=f'THINGS (N={len(things_pool)})', zorder=3)
ax.scatter(lt_pool["residual"], y_lt, c=C_LT, s=60, marker='^',
           edgecolors='k', linewidths=0.5, label=f'LITTLE THINGS (N={len(lt_pool)})', zorder=3)

# Acceptance corridor
ax.axvspan(-0.041, 0.041, alpha=0.15, color='green', label=r'$\pm$10% threshold')
ax.axvline(0, color=C_A0HALF, ls='--', lw=1.5)

# Pooled median
pooled_med = pooled_df["residual"].median()
ax.axvline(pooled_med, color='k', ls='-', lw=2,
           label=f'Pooled median = {pooled_med:+.3f} dex')

ax.set_xlabel(r'$\Delta \log_{10}\, g(R_t)$ [dex]')
ax.set_yticks([1, 2])
ax.set_yticklabels(['THINGS', 'LITTLE\nTHINGS'])
ax.set_ylim(0.3, 2.7)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.set_title('Pooled Confirmatory Test (N=22): FAIL')

# Annotate p-value
ax.text(0.95, 0.05, 'Wilcoxon p = 0.166',
        transform=ax.transAxes, ha='right', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

save_fig(fig, "fig07_pooled_test")
plt.show()

Saved: fig07_pooled_test.pdf, fig07_pooled_test.png


C:\Users\schneider\AppData\Local\Temp\ipykernel_122408\900328913.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Tables

Tables are rendered as formatted DataFrames below. LaTeX versions will be
generated when writing the manuscript.

In [10]:
print("Table 1: Cross-Dataset Summary")
print("=" * 80)
display_cols = ["Dataset", "N_valid_g", "Median_g_mks", "Offset_from_a0half_pct",
                "Scatter_raw_dex", "Scatter_residual_dex", "Median_residual_dex",
                "Wilcoxon_p", "Outcome"]
cross_summary[display_cols]

Table 1: Cross-Dataset Summary


,Dataset,N_valid_g,Median_g_mks,Offset_from_a0half_pct,Scatter_raw_dex,Scatter_residual_dex,Median_residual_dex,Wilcoxon_p,Outcome
0,SPARC,98,6.512157e-11,8.535945,0.408911,0.277452,-0.003984,0.653964,BASELINE (anchored)
1,THINGS,8,8.734848e-11,45.574517,0.345517,0.210477,0.049291,0.546875,INCONCLUSIVE
2,LITTLE THINGS,14,1.656786e-11,-72.388098,0.280099,0.245250,-0.241669,0.049438,REJECTED


In [11]:
print("Table 2: Sample Demographics")
print("=" * 80)
demographics

Table 2: Sample Demographics


,Dataset,N_valid_g,Median_log_Mbar,Median_Mbar_Msun,Min_log_Mbar,Max_log_Mbar,Median_g_Rt_mks
0,SPARC,98,9.763214,5.797149e+09,7.690728,11.422913,6.512157e-11
1,THINGS,8,10.088855,1.227029e+10,8.593563,11.065481,8.734848e-11
2,LITTLE THINGS,14,8.336338,2.169394e+08,6.391288,9.178687,1.656786e-11


In [12]:
print("Table 3: Constrained Model Summary")
print("=" * 80)
constrained_summary

Table 3: Constrained Model Summary


,Dataset,N_total,N_converged,No_solution_rate_pct,Median_delta_BIC,N_free_wins,N_constrained_wins
0,SPARC,123,31,74.796748,96.880187,30,1
1,THINGS,17,4,76.470588,456.858306,3,1
2,LITTLE THINGS,20,20,0.000000,309.295636,19,1


In [13]:
print("Table 5: Per-Galaxy THINGS Results")
print("=" * 80)
things_display = things_df[["galaxy_id", "omega", "Rt", "R_max", "Rt_over_Rmax",
                             "g_Rt_mks", "M_bar", "residual"]].copy()
things_display["g_Rt_mks"] = things_display["g_Rt_mks"].map(lambda x: f"{x:.2e}")
things_display["M_bar"] = things_display["M_bar"].map(lambda x: f"{x:.2e}")
things_display

Table 5: Per-Galaxy THINGS Results


,galaxy_id,omega,Rt,R_max,Rt_over_Rmax,g_Rt_mks,M_bar,residual
0,DDO154_THINGS,12.094323,4.821186,5.856350,0.823241,1.48e-11,3.92e+08,-0.325102
1,NGC2366_THINGS,12.205639,4.776240,5.992582,0.797025,2.23e-11,9.79e+08,-0.241823
2,NGC2403_THINGS,16.765856,6.532926,17.648753,0.370164,7.67e-11,9.28e+09,0.062804
3,NGC2841_THINGS,32.543975,5.293493,51.610792,0.102566,5.62e-10,1.16e+11,0.666071
4,NGC2976_THINGS,35.613874,0.470384,2.247643,0.209279,6.23e-11,1.91e+09,0.135373
5,NGC3198_THINGS,13.374995,7.299758,37.733983,0.193453,9.80e-11,3.36e+10,0.035777
6,NGC3521_THINGS,11.860418,13.520034,17.694713,0.764072,1.05e-10,4.79e+10,0.029760
7,NGC3621_THINGS,21.854827,4.493493,25.770000,0.174369,1.27e-10,1.62e+10,0.222133


In [14]:
print("Table 6: Per-Galaxy LITTLE THINGS Results")
print("=" * 80)
lt_display = lt_df[["galaxy_id", "omega", "Rt", "R_max", "Rt_over_Rmax",
                     "g_Rt_mks", "M_bar", "Mstar_source", "residual"]].copy()
lt_display["g_Rt_mks"] = lt_display["g_Rt_mks"].map(lambda x: f"{x:.2e}")
lt_display["M_bar"] = lt_display["M_bar"].map(lambda x: f"{x:.2e}")
lt_display

Table 6: Per-Galaxy LITTLE THINGS Results


,galaxy_id,omega,Rt,R_max,Rt_over_Rmax,g_Rt_mks,M_bar,Mstar_source,residual
0,DDO_101_LITTLE_THINGS,200.000000,0.301914,2.320001,0.130135,1.01e-10,1.12e+08,MstarSED,0.637524
1,DDO_126_LITTLE_THINGS,15.704786,2.039266,3.990000,0.511094,1.36e-11,2.34e+08,MstarSED,-0.309013
2,DDO_133_LITTLE_THINGS,37.987524,0.975264,3.479999,0.280248,2.64e-11,2.01e+08,MstarSED,-0.004428
3,DDO_154_LITTLE_THINGS,34.163320,1.104753,7.889999,0.140019,2.12e-11,4.77e+08,MstarSED,-0.190245
4,DDO_210_LITTLE_THINGS,46.664013,0.183322,0.310000,0.591362,1.15e-11,2.46e+06,MstarSED,0.088779
5,DDO_216_LITTLE_THINGS,80.708285,0.100000,1.120001,0.089286,6.53e-12,2.16e+07,MstarSED,-0.380770
6,DDO_43_LITTLE_THINGS,30.839446,0.757109,4.190000,0.180694,1.15e-11,3.09e+08,gas-only,-0.411648
7,DDO_52_LITTLE_THINGS,32.993192,1.608395,5.430000,0.296205,2.77e-11,4.98e+08,MstarSED,-0.077320
8,DDO_70_LITTLE_THINGS,43.613272,0.764170,1.999999,0.382085,2.76e-11,7.01e+07,MstarSED,0.122858
9,DDO_87_LITTLE_THINGS,16.562886,3.208522,7.390001,0.434171,1.37e-11,4.20e+08,MstarSED,-0.365114


## Output Summary

### Figures saved to `manuscript/figures/`
| Figure | File | Description |
|--------|------|-------------|
| 1 | `fig01_rt_model_example` | RT model example fit (NGC 2403) |
| 2 | `fig02_sparc_grt_distribution` | SPARC g(Rt) histogram |
| 3 | `fig03_grt_vs_mbar_all` | **Key figure**: g(Rt) vs M_bar, all datasets |
| 4 | `fig04_btfr_residuals` | BTFR residual histograms (3-panel) |
| 5 | `fig05_constrained_bic` | Constrained model BIC comparison |
| 6 | `fig06_cross_pipeline` | Cross-pipeline validation scatter |
| 7 | `fig07_pooled_test` | Pooled test strip plot |